In [2]:
import pandas as pd
import numpy as np
from time import perf_counter
from datasets import load_dataset
from memory_profiler import memory_usage
from tqdm import tqdm

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import DistilBertTokenizer, DistilBertModel
from sklearn.preprocessing import MultiLabelBinarizer


In [3]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
PRETRAINED_MODEL_NAME = 'distilbert-base-uncased-finetuned-sst-2-english'
MAX_LEN = 128
BATCH_SIZE = 16
EPOCHS = 3
LEARNING_RATE = 2e-5 

print(f"Using device: {DEVICE}")

Using device: cuda


In [4]:
ds = load_dataset("TimSchopf/arxiv_categories", "default")

train_df = ds['train'].to_pandas()
val_df = ds['validation'].to_pandas()
test_df = ds['test'].to_pandas()

train_df

,id,title,abstract,categories,creation_date
0,2204.14117,A Comparative Study of Meter Detection Methods...,In order to read meter values from a camera on...,[Computer Science Archive->cs.CV],2022-04-24 13:59:57+00:00
1,2305.19887,The Markov chain embedding problem in a low ju...,We consider the problem of finding the transit...,[Mathematics Archive->math.PR],2023-05-31 14:24:25+00:00
2,0910.5857,Chaotic Transport and Chronology of Complex As...,We present a transport model that describes th...,[Physics Archive->astro-ph->astro-ph.EP],2009-10-30 12:34:26+00:00
3,1801.10207,FITing-Tree: A Data-aware Index Structure,Index structures are one of the most important...,[Computer Science Archive->cs.DB],2018-01-30 20:22:53+00:00
4,0803.0849,The Universal Cardinal Ordering of Fixed Points,"We present the theorem which determines, by a ...",[Physics Archive->nlin->nlin.CD],2008-03-06 12:55:48+00:00
...,...,...,...,...,...
163163,1805.11049,Induced Chern-Simons modified gravity at finit...,We calculate the linearized four-dimensional g...,"[Physics Archive->gr-qc, Physics Archive->hep-...",2018-05-28 17:00:59+00:00
163164,1907.11966,Small Time Behavior and Summability for the Sc...,We consider the Carleson's problem regarding s...,[Mathematics Archive->math.AP],2019-07-27 18:59:04+00:00
163165,1510.08071,GM2Calc: Precise MSSM prediction for $(g - 2)$...,"We present GM2Calc, a public C++ program for t...",[Physics Archive->hep->hep-ph],2015-10-27 20:09:29+00:00
163166,1803.01475,"The Fu-Yau equation on compact astheno-K\""ahle...","In this paper, we study the Fu-Yau equation on...","[Mathematics Archive->math.AP, Mathematics Arc...",2018-03-05 02:54:16+00:00


In [5]:
train_df = train_df.rename(columns={'title': 'text'})
val_df = val_df.rename(columns={'title': 'text'})
test_df = test_df.rename(columns={'title': 'text'})

train_df = train_df.rename(columns={'categories': 'labels'})
val_df = val_df.rename(columns={'categories': 'labels'})
test_df = test_df.rename(columns={'categories': 'labels'})

In [6]:
allowed_categories = ["cs.AI", "cs.CL", "stat.ML", "math.OC", "cs.LG"]

def clean_element(lst):
    final = []
    for elem in lst:
        clean = elem.split('->')[-1]
        final.append(clean)
    return final

train_df['labels'] = train_df['labels'].apply(clean_element)
val_df['labels'] = val_df['labels'].apply(clean_element)
test_df['labels'] = test_df['labels'].apply(clean_element)

In [7]:
train_df = train_df[train_df['labels'].apply(lambda cats: all(c in allowed_categories for c in cats))]
test_df = test_df[test_df['labels'].apply(lambda cats: all(c in allowed_categories for c in cats))]
val_df = val_df[val_df['labels'].apply(lambda cats: all(c in allowed_categories for c in cats))]

train_df = train_df[train_df['labels'].apply(len) > 0]
test_df = test_df[test_df['labels'].apply(len) > 0]
val_df = val_df[val_df['labels'].apply(len) > 0]

In [8]:
train_df.drop(columns=['id','abstract','creation_date'], inplace=True)
test_df.drop(columns=['id','abstract','creation_date'], inplace=True)
val_df.drop(columns=['id','abstract','creation_date'], inplace=True)

train_df.reset_index(drop=True, inplace=True)
test_df.reset_index(drop=True, inplace=True)
val_df.reset_index(drop=True, inplace=True)

train_df

,text,labels
0,Upper and Lower Bounds for Large Scale Multist...,[math.OC]
1,Binary Classification: Counterbalancing Class ...,[cs.LG]
2,Smooth Optimization with Approximate Gradient,[math.OC]
3,An AI-powered Smart Routing Solution for Payme...,[cs.AI]
4,A linearly convergent method for solving high-...,[math.OC]
...,...,...
10141,Simple Question Answering with Subgraph Rankin...,"[cs.CL, cs.LG, stat.ML]"
10142,"Fire Now, Fire Later: Alarm-Based Systems for ...","[cs.AI, cs.LG, stat.ML]"
10143,NSP-BERT: A Prompt-based Few-Shot Learner Thro...,"[cs.AI, cs.CL]"
10144,Near-optimal bounds for phase synchronization,[math.OC]


In [9]:
mlb = MultiLabelBinarizer()

train_labels_binarized = mlb.fit_transform(train_df['labels'])
val_labels_binarized = mlb.transform(val_df['labels'])
test_labels_binarized = mlb.transform(test_df['labels'])

train_labels_df = pd.DataFrame(train_labels_binarized, columns=mlb.classes_)
val_labels_df = pd.DataFrame(val_labels_binarized, columns=mlb.classes_)
test_labels_df = pd.DataFrame(test_labels_binarized, columns=mlb.classes_)

train_df = pd.concat([train_df, train_labels_df], axis=1)
val_df = pd.concat([val_df, val_labels_df], axis=1)
test_df = pd.concat([test_df, test_labels_df], axis=1)

train_df = train_df.drop(columns=['labels'])
val_df = val_df.drop(columns=['labels'])
test_df = test_df.drop(columns=['labels'])

train_df

,text,cs.AI,cs.CL,cs.LG,math.OC,stat.ML
0,Upper and Lower Bounds for Large Scale Multist...,0,0,0,1,0
1,Binary Classification: Counterbalancing Class ...,0,0,1,0,0
2,Smooth Optimization with Approximate Gradient,0,0,0,1,0
3,An AI-powered Smart Routing Solution for Payme...,1,0,0,0,0
4,A linearly convergent method for solving high-...,0,0,0,1,0
...,...,...,...,...,...,...
10141,Simple Question Answering with Subgraph Rankin...,0,1,1,0,1
10142,"Fire Now, Fire Later: Alarm-Based Systems for ...",1,0,1,0,1
10143,NSP-BERT: A Prompt-based Few-Shot Learner Thro...,1,1,0,0,0
10144,Near-optimal bounds for phase synchronization,0,0,0,1,0


In [10]:
class MultiLabelClassificationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        """
        Args:
            texts: List or array of text samples
            labels: 2D array of shape (num_samples, num_classes) with binary indicators (0 or 1)
            tokenizer: Pretrained tokenizer (e.g., DistilBertTokenizer)
            max_len: Maximum sequence length
        """
        self.texts = texts
        self.labels = labels  # Shape: (num_samples, num_classes)
        self.tokenizer = tokenizer
        self.max_len = max_len
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]  # Shape: (num_classes,)
        
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        
        return {
            'text': text,
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.float)  # Binary vector for multilabel
        }

In [11]:
class DistilBertForMultiLabelClassification(nn.Module):
    def __init__(self, num_classes):
        super(DistilBertForMultiLabelClassification, self).__init__()
        self.distilbert = DistilBertModel.from_pretrained(PRETRAINED_MODEL_NAME)
        self.pre_classifier = nn.Linear(768, 768)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(768, num_classes)  # Output logits for each class
    
    def forward(self, input_ids, attention_mask):
        outputs = self.distilbert(input_ids=input_ids, attention_mask=attention_mask)
        hidden_state = outputs[0][:, 0]  # CLS token
        pooled_output = self.pre_classifier(hidden_state)
        pooled_output = nn.ReLU()(pooled_output)
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)
        return logits  # Return raw logits for BCEWithLogitsLoss

In [12]:
def get_metrics(y_true, y_pred):

    acc = accuracy_score(y_true, y_pred)

    precisions, recalls, f1s, supports = precision_recall_fscore_support(y_true, y_pred)

    return acc, precisions, recalls, f1s

In [ ]:
def train_model(model, train_dataloader, val_dataloader, optimizer, criterion, scheduler=None, epochs=EPOCHS):
    best_val_loss = float('inf')
    start_train = perf_counter()
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        train_preds = []
        train_true = []
        
        progress_bar = tqdm(train_dataloader, desc=f'Epoch {epoch + 1}/{epochs}', leave=False)
        for batch in progress_bar:
            optimizer.zero_grad()
            
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)  # Shape: (batch_size, num_classes)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)  # Shape: (batch_size, num_classes)
            loss = criterion(outputs, labels)  # BCEWithLogitsLoss
            train_loss += loss.item()
            
            # Compute binary predictions for each class
            preds = (torch.sigmoid(outputs) > 0.5).float().cpu().numpy()  # Shape: (batch_size, num_classes)
            train_preds.extend(preds)
            train_true.extend(labels.cpu().numpy())
            
            loss.backward()
            optimizer.step()
            progress_bar.set_postfix({'loss': loss.item()})
        
        if scheduler:
            scheduler.step()
            
        train_loss /= len(train_dataloader)
        train_true = np.array(train_true)  # Shape: (num_samples, num_classes)
        train_preds = np.array(train_preds)  # Shape: (num_samples, num_classes)
        
        train_acc, train_precisions, train_recalls, train_f1s = get_metrics(train_true, train_preds)
        
        start_val = perf_counter()
        model.eval()
        val_loss = 0
        val_preds = []
        val_true = []
        with torch.no_grad():
            for batch in tqdm(val_dataloader, desc="Validation", leave=False):
                input_ids = batch['input_ids'].to(DEVICE)
                attention_mask = batch['attention_mask'].to(DEVICE)
                labels = batch['labels'].to(DEVICE)
                
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                
                preds = (torch.sigmoid(outputs) > 0.5).float().cpu().numpy()
                val_preds.extend(preds)
                val_true.extend(labels.cpu().numpy())
        
        val_time = perf_counter() - start_val
        
        val_loss /= len(val_dataloader)
        val_true = np.array(val_true)  # Shape: (num_samples, num_classes)
        val_preds = np.array(val_preds)  # Shape: (num_samples, num_classes)
        
        val_acc, val_precisions, val_recalls, val_f1s = get_metrics(val_true, val_preds)
        
        print(f"Epoch {epoch + 1}/{epochs} - Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f}, F1: {train_f1s}, Prec: {train_precisions}, Recall: {train_recalls}")
        print(f"Epoch {epoch + 1}/{epochs} - Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f}, F1: {val_f1s}, Prec: {val_precisions}, Recall: {val_recalls}, Val Time: {val_time:.2f} sec")
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), 'results/bert_multilabel3.pt')
            print("Model saved!")
    
    total_train_time = perf_counter() - start_train
    print(f"Total Training Time: {total_train_time:.2f} seconds")
    
    return train_acc, train_precisions, train_recalls, train_f1s, val_acc, val_precisions, val_recalls, val_f1s, total_train_time, val_time

In [14]:
def evaluate_model(model, test_dataloader):
    model.eval()
    predictions = []
    true_labels = []
    classification_times = []
    
    start_test = perf_counter()
    
    with torch.no_grad():
        for batch in tqdm(test_dataloader, desc="Testing"):
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)  # Shape: (batch_size, num_classes)
            
            for i in range(input_ids.size(0)):
                input_id = input_ids[i].unsqueeze(0)
                attention_mask_sample = attention_mask[i].unsqueeze(0)
                label = labels[i].cpu().numpy()  # Shape: (num_classes,)
                
                start_time = perf_counter()
                
                output = model(input_ids=input_id, attention_mask=attention_mask_sample)  # Shape: (1, num_classes)
                pred = (torch.sigmoid(output) > 0.5).float().cpu().numpy()[0]  # Shape: (num_classes,)
                
                predictions.append(pred)
                true_labels.append(label)
                classification_times.append(perf_counter() - start_time)
    
    total_test_time = perf_counter() - start_test
    print(f"Test Time: {total_test_time:.2f} seconds")
    
    predictions = np.array(predictions)  # Shape: (num_samples, num_classes)
    true_labels = np.array(true_labels)  # Shape: (num_samples, num_classes)
    
    acc, precisions, recalls, f1s = get_metrics(true_labels, predictions)
    
    print("Test Metrics:")
    print("Accuracy:", acc)
    print("F1s:", f1s)
    print("Precisions:", precisions)
    print("Recalls:", recalls)
    
    return predictions, true_labels

In [15]:
train_texts = train_df['text'].values
train_labels = train_df.drop(columns=['text']).values

val_texts = val_df['text'].values
val_labels = val_df.drop(columns=['text']).values

test_texts = test_df['text'].values
test_labels = test_df.drop(columns=['text']).values

tokenizer = DistilBertTokenizer.from_pretrained(PRETRAINED_MODEL_NAME)

# Use MultiClassClassificationDataset instead of BinaryClassificationDataset
train_dataset = MultiLabelClassificationDataset(train_texts, train_labels, tokenizer, MAX_LEN)
val_dataset = MultiLabelClassificationDataset(val_texts, val_labels, tokenizer, MAX_LEN)
test_dataset = MultiLabelClassificationDataset(test_texts, test_labels, tokenizer, MAX_LEN)

train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

seeds = [2,3,5]

num_classes = len(allowed_categories)

avg_train_acc = 0
avg_train_precs = np.zeros(num_classes)
avg_train_recalls = np.zeros(num_classes)
avg_train_f1s = np.zeros(num_classes)
avg_max_memory_usage_train = 0
avg_max_vram_usage_train = 0
avg_total_train_time = 0

avg_val_acc = 0
avg_val_precs = np.zeros(num_classes)
avg_val_recalls = np.zeros(num_classes)
avg_val_f1s = np.zeros(num_classes)
avg_total_val_time = 0

avg_test_acc = 0
avg_test_precs = np.zeros(num_classes)
avg_test_recalls = np.zeros(num_classes)
avg_test_f1s = np.zeros(num_classes)
avg_max_memory_usage_test = 0
avg_max_vram_usage_test = 0
avg_total_test_time = 0

for seed in seeds:
    torch.manual_seed(seed)
    model = DistilBertForMultiLabelClassification(num_classes)
    model = model.to(DEVICE)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.BCEWithLogitsLoss()  # For multi-label

    if torch.cuda.is_available():
        torch.cuda.reset_max_memory_allocated()

    max_memory_usage_train, retval = memory_usage(
        (train_model, (model, train_dataloader, val_dataloader, optimizer, criterion), {'epochs': EPOCHS}),
        max_usage=True,
        retval=True
    )

    max_vram_usage_train = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else 0

    (train_acc, train_precisions, train_recalls, train_f1s,
     val_acc, val_precisions, val_recalls, val_f1s,
     total_train_time, val_time) = retval

    model.load_state_dict(torch.load('results/bert_multilabel3.pt'))

    if torch.cuda.is_available():
        torch.cuda.reset_max_memory_allocated()

    start = perf_counter()
    max_memory_usage_test, retval = memory_usage(
        (evaluate_model, (model, test_dataloader), {}),
        max_usage=True,
        retval=True
    )
    total_time_test = perf_counter() - start

    max_vram_usage_test = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else 0

    predictions, true_labels = retval

    test_acc, test_precisions, test_recalls, test_f1s = get_metrics(true_labels, predictions)

    avg_train_acc += train_acc
    avg_train_precs += train_precisions
    avg_train_recalls += train_recalls
    avg_train_f1s += train_f1s
    avg_max_memory_usage_train += max_memory_usage_train
    avg_max_vram_usage_train += max_vram_usage_train
    avg_total_train_time += total_train_time

    avg_val_acc += val_acc
    avg_val_precs += val_precisions
    avg_val_recalls += val_recalls
    avg_val_f1s += val_f1s
    avg_total_val_time += val_time

    avg_test_acc += test_acc
    avg_test_precs += test_precisions
    avg_test_recalls += test_recalls
    avg_test_f1s += test_f1s
    avg_max_memory_usage_test += max_memory_usage_test
    avg_max_vram_usage_test += max_vram_usage_test
    avg_total_test_time += total_time_test

avg_train_acc /= len(seeds)
avg_train_precs /= len(seeds)
avg_train_recalls /= len(seeds)
avg_train_f1s /= len(seeds)
avg_max_memory_usage_train /= len(seeds)
avg_max_vram_usage_train /= len(seeds)
avg_total_train_time /= len(seeds)

avg_val_acc /= len(seeds)
avg_val_precs /= len(seeds)
avg_val_recalls /= len(seeds)
avg_val_f1s /= len(seeds)
avg_total_val_time /= len(seeds)

avg_test_acc /= len(seeds)
avg_test_precs /= len(seeds)
avg_test_recalls /= len(seeds)
avg_test_f1s /= len(seeds)
avg_max_memory_usage_test /= len(seeds)
avg_max_vram_usage_test /= len(seeds)
avg_total_test_time /= len(seeds)

avg_classification_time = avg_total_test_time / len(test_texts)

avg_classification_time

c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:343: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(


Epoch 1/3 - Train Loss: 0.4185, Acc: 0.3674, F1: [0.18069689 0.81089109 0.75294338 0.66537468 0.34256521], Prec: [0.54158965 0.86819788 0.73683248 0.75292398 0.543052  ], Recall: [0.10843819 0.76068111 0.76977455 0.59606481 0.25019639]
Epoch 1/3 - Val Loss: 0.3444, Acc: 0.4581, F1: [0.45506692 0.90036014 0.79803761 0.77083333 0.43534483], Prec: [0.63636364 0.87209302 0.85614035 0.87573964 0.69655172], Recall: [0.35416667 0.93052109 0.74732006 0.68837209 0.31661442], Val Time: 2.37 sec
Model saved!


Epoch 2/3 - Train Loss: 0.3212, Acc: 0.5031, F1: [0.458413   0.91034267 0.82511299 0.82958004 0.5728792 ], Prec: [0.64709852 0.92027839 0.8493123  0.87483954 0.62081614], Recall: [0.35492228 0.9006192  0.80225449 0.78877315 0.53181461]
Epoch 2/3 - Val Loss: 0.3273, Acc: 0.4984, F1: [0.44664032 0.90931677 0.80932556 0.81278539 0.56752137], Prec: [0.66470588 0.91044776 0.88686131 0.79820628 0.62406015], Recall: [0.33630952 0.90818859 0.74425727 0.82790698 0.52037618], Val Time: 2.38 sec
Model saved!


Epoch 3/3 - Train Loss: 0.2703, Acc: 0.5689, F1: [0.55623444 0.94180059 0.86445163 0.88515742 0.65440736], Prec: [0.71578334 0.94935514 0.88750252 0.91848164 0.66625967], Recall: [0.45484826 0.93436533 0.84256783 0.85416667 0.64296936]
Epoch 3/3 - Val Loss: 0.3249, Acc: 0.5016, F1: [0.47927928 0.90721649 0.83458647 0.82926829 0.60066007], Prec: [0.60730594 0.94369973 0.81979321 0.87179487 0.63414634], Recall: [0.39583333 0.87344913 0.84992343 0.79069767 0.57053292], Val Time: 2.38 sec
Model saved!
Total Training Time: 189.02 seconds


C:\Users\Rafael\AppData\Local\Temp\ipykernel_3772\12187452.py:70: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('results/bert_multilabel2.pt

...

C:\Users\Rafael\AppData\Local\Temp\ipykernel_3772\12187452.py:70: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('results/bert_multilabel2.pt

Epoch 3/3 - Train Loss: 0.2684, Acc: 0.5668, F1: [0.55502609 0.93710889 0.86828982 0.88782244 0.65514501], Prec: [0.71730205 0.94718533 0.88924494 0.92154421 0.66278135], Recall: [0.45262768 0.92724458 0.84829958 0.85648148 0.64768264]
Epoch 3/3 - Val Loss: 0.3256, Acc: 0.5126, F1: [0.44913628 0.91831683 0.82389449 0.8441247  0.6009539 ], Prec: [0.63243243 0.91604938 0.83490566 0.87128713 0.60967742], Recall: [0.34821429 0.92059553 0.81316998 0.81860465 0.59247649], Val Time: 2.53 sec
Total Training Time: 197.57 seconds


Testing: 100%|██████████| 80/80 [00:07<00:00, 10.95it/s]

Test Time: 7.31 seconds
Test Metrics:
Accuracy: 0.4921259842519685
F1s: [0.51090343 0.89960887 0.79935275 0.83135392 0.59322034]
Precisions: [0.54125413 0.95303867 0.85172414 0.85365854 0.64102564]
Recalls: [0.48377581 0.85185185 0.75304878 0.81018519 0.55205047]


0.006084136929139348

In [17]:
# save results to txt
with open("results/bert_multilabel3.txt", "w") as f:
    f.write(f"Average Train Accuracy: {avg_train_acc}\n")
    f.write(f"Average Train Precisions: {avg_train_precs}\n")
    f.write(f"Average Train Recalls: {avg_train_recalls}\n")
    f.write(f"Average Train F1s: {avg_train_f1s}\n")
    f.write(f"Average Max Memory Usage Train: {avg_max_memory_usage_train}\n")
    f.write(f"Average Max VRAM Usage Train: {avg_max_vram_usage_train}\n")
    f.write(f"Average Total Train Time: {avg_total_train_time}\n")
    f.write("\n")
    f.write(f"Average Val Accuracy: {avg_val_acc}\n")
    f.write(f"Average Val Precisions: {avg_val_precs}\n")
    f.write(f"Average Val Recalls: {avg_val_recalls}\n")
    f.write(f"Average Val F1s: {avg_val_f1s}\n")
    f.write(f"Average Total Val Time: {avg_total_val_time}\n")
    f.write("\n")
    f.write(f"Average Test Accuracy: {avg_test_acc}\n")
    f.write(f"Average Test Precisions: {avg_test_precs}\n")
    f.write(f"Average Test Recalls: {avg_test_recalls}\n")
    f.write(f"Average Test F1s: {avg_test_f1s}\n")
    f.write(f"Average Max Memory Usage Test: {avg_max_memory_usage_test}\n")
    f.write(f"Average Max VRAM Usage Test: {avg_max_vram_usage_test}\n")
    f.write(f"Average Total Test Time: {avg_total_test_time}\n")
    f.write("\n")
    f.write(f"Average Classification Time: {avg_classification_time}\n")
    f.write(f"Lines classified {len(test_texts)}\n")

    f.close()